In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
spark.sql(f"""
INSERT OVERWRITE TABLE {catalogo}.{esquema_sink}.orders_transformed
 
WITH payments_sum AS (
    -- Un order_id puede tener varias filas de pago; se agrega el monto total
    -- para no duplicar filas al unir con items.
    SELECT
        order_id,
        SUM(payment_value) AS payment_value
    FROM {catalogo}.{esquema_source}.payments
    GROUP BY order_id
),
 
payments_primary AS (
    -- Se elige el método de pago "principal" (mayor payment_value por order_id).
    SELECT
        order_id,
        payment_type,
        payment_installments
    FROM {catalogo}.{esquema_source}.payments
    QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY payment_value DESC) = 1
),
 
payments_agg AS (
    SELECT
        ps.order_id,
        ps.payment_value,
        pp.payment_type,
        pp.payment_installments
    FROM payments_sum AS ps
    JOIN payments_primary AS pp
        ON ps.order_id = pp.order_id
),
 
geo_dedup AS (
    -- La bronze de geolocation trae múltiples lat/lng por zip prefix;
    -- se promedia a un solo punto representativo por zip.
    SELECT
        geolocation_zip_code_prefix,
        AVG(geolocation_lat)  AS geolocation_lat,
        AVG(geolocation_lng)  AS geolocation_lng,
        FIRST(geolocation_city)  AS geolocation_city,
        FIRST(geolocation_state) AS geolocation_state
    FROM {catalogo}.{esquema_source}.geolocation
    GROUP BY geolocation_zip_code_prefix
)
 
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_zip_code_prefix,
    c.customer_city,
    c.customer_state,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    pa.payment_type,
    pa.payment_installments,
    pa.payment_value,
    i.product_id,
    p.product_category_name,
    t.product_category_name_english,
    i.order_item_id,
    i.seller_id,
    i.shipping_limit_date,
    i.price,
    i.freight_value,
    s.seller_zip_code_prefix,
    s.seller_city,
    s.seller_state,
    g.geolocation_zip_code_prefix,
    g.geolocation_lat,
    g.geolocation_lng,
    g.geolocation_city,
    g.geolocation_state,
    current_timestamp() AS ingestion_date
FROM {catalogo}.{esquema_source}.orders AS o
INNER JOIN {catalogo}.{esquema_source}.items AS i
    ON o.order_id = i.order_id
LEFT JOIN {catalogo}.{esquema_source}.customers AS c
    ON o.customer_id = c.customer_id
LEFT JOIN {catalogo}.{esquema_source}.products AS p
    ON i.product_id = p.product_id
LEFT JOIN {catalogo}.{esquema_source}.translation AS t
    ON p.product_category_name = t.product_category_name
LEFT JOIN {catalogo}.{esquema_source}.sellers AS s
    ON i.seller_id = s.seller_id
LEFT JOIN payments_agg AS pa
    ON o.order_id = pa.order_id
LEFT JOIN geo_dedup AS g
    ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
""")